# 3. Agent demo: MCP tools and Langfuse observability

В этой тетрадке проверяется последняя часть задания: подключение поиска как MCP-инструмента к агенту.

На предыдущих этапах уже были подготовлены:

- нормализованные JSON-файлы с темами исторических эссе;
- enriched chunks;
- локальный Qdrant index;
- search service с semantic search, metadata search и metadata-aware reranking.

Теперь эти возможности публикуются через MCP server. Агент получает MCP tools и вызывает их для поиска top-k фрагментов.

Дополнительно подключается Langfuse, чтобы видеть trace выполнения: вызовы LLM, выбор tools, параметры MCP-вызовов, результаты поиска и финальный ответ агента.

In [1]:
from hw_rag_mcp.settings import get_settings, REPO_ROOT

settings = get_settings()
settings.print_summary()

✓ Конфигурация загружена
  Env file:             /Users/dmitrijkanevskij/VS_CodeProjects/data_science/agents/courses/agents_hse1/hw_rag_mcp/.env
  Repo root:            /Users/dmitrijkanevskij/VS_CodeProjects/data_science/agents/courses/agents_hse1/hw_rag_mcp
  LLM:                  GigaChat-2-MAX
  Embeddings:           EmbeddingsGigaR
  Qdrant path:          /Users/dmitrijkanevskij/VS_CodeProjects/data_science/agents/courses/agents_hse1/hw_rag_mcp/data/vectorstore/qdrant
  Qdrant collection:    history_essay_chunks
  Langfuse base URL:    https://cloud.langfuse.com
  GigaChat credentials: заданы
  Langfuse credentials: заданы


## 2. Langfuse connection

Langfuse используется для наблюдаемости агентного запуска: LLM-вызовы, tool calls, параметры tools и итоговый ответ.

Если `auth_check()` возвращает `True`, значит ключи и host настроены корректно.

In [3]:
from langfuse import Langfuse, get_client
from langfuse.langchain import CallbackHandler

langfuse = Langfuse(
    public_key=settings.langfuse_public_key,
    secret_key=settings.langfuse_secret_key,
    base_url=settings.langfuse_base_url,
)

assert langfuse.auth_check(), "Langfuse auth failed. Check LANGFUSE_* variables."

langfuse_handler = CallbackHandler()

print("✅ Langfuse connected")

/Users/dmitrijkanevskij/VS_CodeProjects/data_science/agents/courses/agents_hse1/hw_rag_mcp/.venv/lib/python3.13/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


✅ Langfuse connected


## 3. MCP client

Подключаем MCP-сервер `history-essay-rag` через stdio.

Сервер запускается командой:

```bash
uv run python -m hw_rag_mcp.mcp.server
```

Внутри MCP-сервера опубликованы tools:

- `semantic_search`
- `search_by_quote_author`
- `search_by_interpretation_type`
- `boosted_search`
- `list_interpretation_types`
- `infer_interpretation_type_with_sampling`
- `sampled_boosted_search`

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "history_essay_rag": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "hw_rag_mcp.mcp_server",
            ],
            # Some versions of langchain-mcp-adapters support cwd.
            # If your version does not, remove this key and run the notebook from project root.
            "cwd": str(REPO_ROOT),
        }
    }
)

tools = await mcp_client.get_tools()

print(f"Loaded tools: {len(tools)}")
for tool in tools:
    print("-", tool.name)

Loaded tools: 3
- semantic_search
- search_by_quote_author
- search_by_query_and_interpretation_type


## 4. Manual metadata tools

Проверим все три MCP tools вручную, без агента.

Это нужно для отладки: если tools работают напрямую, значит MCP-сервер корректно стартует, видит Qdrant index, использует правильную embedding model и возвращает результаты с metadata.

Проверяем три сценария:

1. обычный semantic search;
2. поиск по автору цитаты;
3. semantic search + metadata reranking по типу авторской позиции.

In [4]:
tool_by_name = {tool.name: tool for tool in tools}

semantic_result = await tool_by_name["semantic_search"].ainvoke(
    {
        "query": (
            "Киевская Русь, объединение восточных славян, "
            "формирование древнерусского государства, политическая роль Киева "
            "в IX-X веках."
        )
    }
)

semantic_result

[{'type': 'text',
  'text': '{"query":"Киевская Русь, объединение восточных славян, формирование древнерусского государства, политическая роль Киева в IX-X веках.","results":[{"document_id":"final_2015","chunk_id":"final_2015_topic_001","source":"final_2015.pdf","author":"В.В. Мавродин","content":"«Несмотря на примитивный характер объединения русских земель и племён, Киевская держава в лице её политических деятелей делала великое дело: объединяла восточное славянство в единый государственный организм, сплачивая и тем самым усиливая его, создавая условия для дальнейшего укрепления общности языка, быта, культуры, и обороняла рубежи земель русского народа, рубежи Руси»"},{"document_id":"reg_2022","chunk_id":"reg_2022_topic_001","source":"reg_2022.pdf","author":"А.А. Горский","content":"«Говорить о неустоявшейся структуре властвования, об отсутствии у Киева статуса главного центра нет серьезных оснований. Киев уже был по меньшей мере с начала X в. признанным главным центром Руси, власть пр

In [5]:
author_result = await tool_by_name["search_by_quote_author"].ainvoke(
    {
        "author": "Сталин"
    }
)

author_result

[{'type': 'text',
  'text': '{"query":"Сталин","results":[{"document_id":"final_2018","chunk_id":"final_2018_topic_017","source":"final_2018.pdf","author":"И.В. Сталин","content":"«Уроки войны говорят о том, что Советский строй оказался не только лучшей формой организации экономического и культурного подъема страны в годы мирного строительства, но и лучшей формой мобилизации всех сил народа на отпор врагу в военное время»"},{"document_id":"final_2024","chunk_id":"final_2024_topic_013","source":"final_2024.pdf","author":"И.В. Сталин","content":"«Тегеранская конференция не прошла даром… Одновременно с летними операциями Красной Армии на советско-германском фронте… войска и флот наших союзников совершили невиданную еще в истории по организованности и размаху массовую десантную операцию на побережье Франции и мастерски преодолели укрепления немцев... Успешное осуществление Тегеранского решения не могло не послужить делу упрочения фронта Объединенных наций…»"}]}',
  'id': 'lc_beb6c4ab-5cc9-

In [6]:
interpretation_result = await tool_by_name["search_by_query_and_interpretation_type"].ainvoke(
    {
        "query": (
            "История Великого княжества Владимирского в период татаро-монгольского "
            "нашествия, Александр Невский, политика князя в отношении Орды, "
            "оценка его роли в истории Руси XIII века."
        ),
        "interpretation_type": "negative_assessment",
    }
)

interpretation_result

[{'type': 'text',
  'text': '{"query":"История Великого княжества Владимирского в период татаро-монгольского нашествия, Александр Невский, политика князя в отношении Орды, оценка его роли в истории Руси XIII века.","results":[{"document_id":"reg_2018","chunk_id":"reg_2018_topic_002","source":"reg_2018.pdf","author":"Дж. Феннел","content":"«Вторая половина XIII в. была эпохой постепенно возраставшего татарского давления – нашествия, набегов, унижения. Этот режим татарского гнета укрепился отчасти и в результате политики Александра Невского. Но вина за тяготы татарского господства на Руси лежит и на преемниках Александра, которые без колебаний следовали его примеру, призывая татарские войска на русскую землю для достижения своих политических целей»"},{"document_id":"final_2018","chunk_id":"final_2018_topic_002","source":"final_2018.pdf","author":"Н.И. Костомаров","content":"«В действиях Дмитрия виден ряд промахов. Следуя задаче подчинить Москве русские земли… он не уничтожил силы и самос

### Вывод по ручной проверке MCP tools

Все три MCP tools проверяются отдельно до подключения агента.

`semantic_search` используется для обычного смыслового поиска по исторической теме. В него передаётся развёрнутый semantic query, а не короткое ключевое слово.

`search_by_quote_author` работает как metadata-поиск по автору цитаты. Он не использует semantic search, а проверяет структурированное поле `quote_author`.

`search_by_query_and_interpretation_type` комбинирует semantic search и metadata-aware reranking. Сначала он ищет темы по смыслу, затем поднимает выше результаты с нужным `interpretation_type`.

После этой проверки можно переходить к агенту: агент должен получить те же MCP tools и сам выбрать нужный инструмент по пользовательскому запросу.

## 5. Agent with MCP tools

Теперь подключаем MCP tools к агенту.

System prompt специально ограничивает роль агента: он не должен отвечать на исторический вопрос напрямую. Его задача — вызвать MCP-инструмент поиска и вернуть найденные темы с обязательными metadata: `document_id`, `chunk_id`, `source`, автор цитаты, исторический период и текст темы.

In [5]:
from langchain.agents import create_agent
from langchain_gigachat import GigaChat

llm = GigaChat(
    credentials=settings.gigachat_credentials,
    scope=settings.gigachat_scope,
    model=settings.gigachat_model,
    verify_ssl_certs=False,
    temperature=0.1,
    timeout=60,
)

system_prompt = (
    "Ты агент для поиска тем исторических эссе школьных олимпиад. "
    "Твоя задача — найти подходящие темы в корпусе через доступные инструменты поиска. "
    "Не отвечай на исторический вопрос напрямую и не пиши сочинение. "
    "Используй только результаты инструментов. "
    "Для каждой найденной темы обязательно покажи document_id, chunk_id, source, author и content. "
    "content — это текст найденной темы. "
    "Не добавляй исторические факты от себя и не выдумывай отсутствующие поля."
)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
)

print("✅ Agent created")

✅ Agent created


### Helpers

In [6]:
async def run_agent_query(query: str):
    response = await agent.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": query,
                }
            ]
        },
        config={
            "callbacks": [langfuse_handler],
            "metadata": {
                "langfuse_session_id": "history-essay-rag-mcp-demo",
                "langfuse_user_id": "student",
                "langfuse_tags": ["mcp", "rag", "history-essay", "agent"],
            },
        },
    )

    return response


def print_agent_answer(response):
    messages = response.get("messages", [])

    if not messages:
        print(response)
        return

    last_message = messages[-1]
    content = getattr(last_message, "content", None)

    if content is None and isinstance(last_message, dict):
        content = last_message.get("content")

    print(content)

## 7 Agentic calls
### 7.1 Semantic query

Первый тест — обычный смысловой поиск без автора цитаты и без типа оценки.

Ожидаем, что агент выберет semantic search.

In [9]:
response_1 = await run_agent_query(
    "Найди темы исторических эссе про Киевскую Русь, объединение восточных славян "
    "и формирование древнерусского государства."
)

print_agent_answer(response_1)

Вот несколько подходящих тем из корпуса:

1. **Тема:** «Несмотря на примитивный характер объединения русских земель и племён, Киевская держава в лице её политических деятелей делала великое дело: объединяла восточное славянство в единый государственный организм, сплачивая и тем самым усиливая его, создавая условия для дальнейшего укрепления общности языка, быта, культуры, и обороняла рубежи земель русского народа, рубежи Руси»
   - **document_id:** final_2015
   - **chunk_id:** final_2015_topic_001
   - **source:** final_2015.pdf
   - **author:** В.В. Мавродин

2. **Тема:** «Говорить о неустоявшейся структуре властвования, об отсутствии у Киева статуса главного центра нет серьезных оснований. Киев уже был по меньшей мере с начала X в. признанным главным центром Руси, власть принадлежала княжескому роду, контролировавшему огромную территорию непосредственно и по меньшей мере такую же – через признающих верховенство киевского князя местных князей»
   - **document_id:** reg_2022
   - **ch

### 7.2 Quote author query

Второй тест — поиск по автору цитаты.

Ожидаем, что агент выберет metadata-поиск по `quote_author`, а не обычный semantic search.

In [10]:
response_2 = await run_agent_query(
    "Найди темы исторических эссе, где автор цитаты — Сталин. "
)

print_agent_answer(response_2)

Вот две темы исторических эссе, где автором цитаты является И.В. Сталин:

1. **Document ID:** final_2018  
   **Chunk ID:** final_2018_topic_017  
   **Source:** final_2018.pdf  
   **Author:** И.В. Сталин  
   **Content:** «Уроки войны говорят о том, что Советский строй оказался не только лучшей формой организации экономического и культурного подъема страны в годы мирного строительства, но и лучшей формой мобилизации всех сил народа на отпор врагу в военное время»

2. **Document ID:** final_2024  
   **Chunk ID:** final_2024_topic_013  
   **Source:** final_2024.pdf  
   **Author:** И.В. Сталин  
   **Content:** «Тегеранская конференция не прошла даром… Одновременно с летними операциями Красной Армии на советско-германском фронте… войска и флот наших союзников совершили невиданную еще в истории по организованности и размаху массовую десантную операцию на побережье Франции и мастерски преодолели укрепления немцев... Успешное осуществление Тегеранского решения не могло не послужить делу

### 7.3 Semantic query + interpretation type

Третий тест — смешанный запрос: историческая тема + тип авторской позиции.

Ожидаем, что агент выберет поиск, который совмещает semantic search и metadata-aware reranking по `interpretation_type`.

In [11]:
response_3 = await run_agent_query(
    "Найди темы про Александра Невского с отрицательной или критической оценкой автора. "
    "Покажи document_id, chunk_id и source."
)

print_agent_answer(response_3)

Тема про Александра Невского с отрицательной оценкой автора:

- **document_id**: reg_2018  
- **chunk_id**: reg_2018_topic_002  
- **source**: reg_2018.pdf  
- **author**: Дж. Феннел  
- **content**: «Вторая половина XIII в. была эпохой постепенно возраставшего татарского давления – нашествия, набегов, унижения. Этот режим татарского гнета укрепился отчасти и в результате политики Александра Невского. Но вина за тяготы татарского господства на Руси лежит и на преемниках Александра, которые без колебаний следовали его примеру, призывая татарские войска на русскую землю для достижения своих политических целей»

Также есть другие темы с негативными оценками других деятелей того времени (например, Дмитрия), но конкретно про Александра Невского подходит первая тема.


### 7.4 Comparative query

Четвёртый тест — запрос на сравнительную интерпретацию.

Ожидаем, что агент распознает требование сравнения и использует `interpretation_type = comparative`.

In [12]:
response_4 = await run_agent_query(
    "Найди сравнительные темы про Киевскую Русь и Западную Европу. "
    "Покажи document_id, chunk_id и source."
)

print_agent_answer(response_4)

Вот сравнительные темы про Киевскую Русь и Западную Европу из корпуса:

1. **document_id:** final_2023  
   **chunk_id:** final_2023_topic_001  
   **source:** final_2023.pdf  
   **content:** «В Киевской Руси… монетарная экономика превалировала над натуральным хозяйством. И, в отличие от Запада, не феодальное поместье, а город был главным фактором экономической и социальной эволюции страны. Но если в социально-политическом развитии Руси и Западной Европы в Средние века существовали значительные различия, то имелись также и многочисленные сходства»

2. **document_id:** final_2024  
   **chunk_id:** final_2024_topic_001  
   **source:** final_2024.pdf  
   **content:** «Серьезных оснований видеть в Киевской Руси государство имперского типа нет. Типологически она, следовательно, ближе не Византийской империи и не Франкской империи Карла Великого и его потомков, а моноэтничным европейским государствам Средневековья»

3. **document_id:** final_2022  
   **chunk_id:** final_2022_topic_002  

In [13]:
get_client().flush()
print("✅ Trace flushed to Langfuse")

✅ Trace flushed to Langfuse


## 8. Langfuse trace

После agent/tool вызововов можно открыть Langfuse UI и проверить trace.

In [14]:
print("Open Langfuse UI:", settings.langfuse_base_url)

Open Langfuse UI: https://cloud.langfuse.com


### Вывод по агентным вызовам

На этом шаге мы проверили, что агент получает MCP tools через `langchain-mcp-adapters`, выбирает инструмент поиска и возвращает найденные фрагменты из корпуса.

После упрощения ответа MCP-сервера результат стал заметно стабильнее. Вместо большого payload с большим количеством metadata MCP tools возвращают компактные поля:

- `document_id`;
- `chunk_id`;
- `source`;
- `author`;
- `content`.

Это уменьшило шум в контексте модели и улучшило качество финального ответа: агент стал меньше пересказывать лишние данные и лучше фокусироваться на найденных темах.

По трассам Langfuse видно, что основные сценарии работают:

| Сценарий | Ожидаемое поведение | Фактическое поведение |
|---|---|---|
| Обычный смысловой запрос | semantic search | Сработало корректно |
| Запрос по автору цитаты | поиск по `quote_author` | Сработало корректно |
| Запрос про отрицательную оценку | поиск по query + `negative_assessment` | Сработало корректно |
| Сравнительный запрос | поиск по query + `comparative` | После перехода на более сильную модель стал срабатывать корректнее |

Отдельно видно, что более сильная модель лучше справляется с выбором инструмента и с фильтрацией результатов. Например, в запросе про отрицательную оценку Александра Невского агент не стал возвращать все темы с негативной оценкой подряд, а выбрал наиболее релевантную тему именно про Александра Невского. Это хороший признак: агент не просто механически выводит все результаты tool call, а дополнительно отбирает наиболее подходящие фрагменты.

При этом остаётся небольшое ограничение: несмотря на то что MCP-сервер возвращает поле `author`, модель иногда не выводит его в финальном ответе. Это не ломает сам pipeline, потому что в Langfuse trace и в ответе MCP tool поле присутствует. Но это показывает типичную проблему агентных систем: финальное форматирование ответа LLM не всегда полностью детерминировано. Поэтому для проверки корректности tool call и фактических данных надёжнее смотреть не только текст ответа агента, но и Langfuse trace.

Итог: MCP-интеграция работает, агент вызывает поисковые tools, Qdrant-backed search service возвращает релевантные фрагменты, а Langfuse позволяет проверить реальные tool calls, аргументы и ответы инструментов. Следующий шаг — провести систематическую проверку на eval-наборе из 15–20 запросов.

## 9. Eval dataset

Для проверки агента был создан небольшой eval-набор из 20 запросов. Он лежит в `data/eval/eval_queries.json`

Цель eval — не автоматическая строгая проверка качества, а воспроизводимый набор вопросов для ручной оценки retrieval pipeline:

- агент получает пользовательский запрос;
- выбирает MCP tool;
- получает top-k фрагменты;
- возвращает ответ;
- результат сохраняется в `data/eval/eval_results.json`.

После этого результаты можно вручную проанализировать и оформить в `notebooks/eval_report.md`.

Запускаем все eval-запросы через того же агента, который использовался в демонстрации.

В результатах сохраняем:

- `id`;
- `category`;
- `query`;
- `expected_behavior`;
- `agent_answer`;
- `error`, если запрос упал.

Ручную оценку (`good`, `partial`, `bad`) добавим после просмотра `eval_results.json`.

In [11]:
import json

def extract_agent_answer(response) -> str:
    messages = response.get("messages", [])

    if not messages:
        return str(response)

    last_message = messages[-1]
    content = getattr(last_message, "content", None)

    if content is None and isinstance(last_message, dict):
        content = last_message.get("content")

    return str(content)

eval_queries = json.load(open(f"../{settings.eval_queries_path}"))
eval_results = []

for case in eval_queries:
    print(f"Running {case['id']}: {case['query']}")

    try:
        response = await run_agent_query(case["query"])
        agent_answer = extract_agent_answer(response)

        eval_results.append(
            {
                "id": case["id"],
                "category": case["category"],
                "query": case["query"],
                "expected_behavior": case["expected_behavior"],
                "agent_answer": agent_answer,
                "error": None,
            }
        )

    except Exception as error:
        eval_results.append(
            {
                "id": case["id"],
                "category": case["category"],
                "query": case["query"],
                "expected_behavior": case["expected_behavior"],
                "agent_answer": None,
                "error": {
                    "type": type(error).__name__,
                    "message": str(error),
                },
            }
        )

with open(f"../{settings.eval_results_path}", "w", encoding="utf-8") as file:
    json.dump(eval_results, file, ensure_ascii=False, indent=2)

print(f'Saved {len(eval_results)} eval results to "../{settings.eval_results_path}"')

Running q01: Найди темы исторических эссе про Киевскую Русь, объединение восточных славян и формирование древнерусского государства.
Running q02: Найди темы про реформы Петра I, модернизацию государства, армии, флота и управления.
Running q03: Найди темы про советскую индустриализацию, развитие тяжелой промышленности и социальные последствия политики 1930-х годов.
Running q04: Найди темы про Смутное время, кризис власти, народные движения и восстановление государственности в России начала XVII века.
Running q05: Найди темы исторических эссе, где автор цитаты Сталин.
Running q06: Найди темы исторических эссе, где автор цитаты Карамзин.
Running q07: Найди темы исторических эссе, где автор цитаты Вернадский.
Running q08: Найди темы исторических эссе, где автор цитаты Ключевский.
Running q09: Найди темы про Александра Невского с отрицательной или критической оценкой автора.
Running q10: Найди темы, где автор положительно оценивает роль советского строя или советской власти.
Running q11: На

In [9]:
eval_results

[{'id': 'q01',
  'category': 'happy_semantic',
  'query': 'Найди темы исторических эссе про Киевскую Русь, объединение восточных славян и формирование древнерусского государства.',
  'expected_behavior': 'Должен сработать обычный смысловой поиск. В результатах ожидаются темы про Киевскую Русь, объединение восточных славян, формирование древнерусского государства.',
  'agent_answer': 'Вот несколько подходящих тем из корпуса:\n\n1. **Тема:** «Несмотря на примитивный характер объединения русских земель и племён, Киевская держава в лице её политических деятелей делала великое дело: объединяла восточное славянство в единый государственный организм, сплачивая и тем самым усиливая его, создавая условия для дальнейшего укрепления общности языка, быта, культуры, и обороняла рубежи земель русского народа, рубежи Руси»\n   - **document_id:** final_2015\n   - **chunk_id:** final_2015_topic_001\n   - **source:** final_2015.pdf\n   - **author:** В.В. Мавродин\n\n2. **Тема:** «Говорить о неустоявшейс

In [10]:
with open(f"../{settings.eval_queries_path}", "w", encoding="utf-8") as file:
    json.dump(eval_results, file, ensure_ascii=False, indent=2)

print(f'Saved {len(eval_results)} eval results to "../{settings.eval_queries_path}"')

Saved 1 eval results to "../data/eval/eval_queries.json"


## 10. Итог

В этой тетрадке мы проверили финальную связку проекта:

```text
агент → MCP tools → Qdrant-backed search service → top-k фрагменты
```

Сначала MCP-инструменты были вызваны вручную. Затем те же инструменты были подключены к агенту через `langchain-mcp-adapters`.

Агент смог обращаться к поиску через MCP boundary и возвращать найденные темы с основными полями:

- `document_id`;
- `chunk_id`;
- `source`;
- `author`;
- `content`.

Для финальной версии мы оставили простую и понятную схему из трёх поисковых инструментов:

| MCP tool | Назначение |
|---|---|
| `semantic_search` | обычный смысловой поиск по теме |
| `search_by_quote_author` | поиск по автору цитаты |
| `search_by_query_and_interpretation_type` | поиск по теме с учётом типа авторской позиции |

Отдельная серверная трассировка MCP не настраивалась. Для этого задания достаточно показать, что агент вызывает MCP-инструменты и получает результаты через протокольную границу.

Качество работы агента и retrieval-пайплайна дополнительно проверено на eval-наборе из 20 запросов. Подробный анализ результатов находится в файле:

```text
notebooks/eval_report.md
```

Итог: RAG-индекс построен, MCP-сервер запускается, инструменты поиска работают, агент подключается к MCP и возвращает релевантные фрагменты корпуса.
